In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(ggpubr)
getwd()
dir.create("figures_10xMouse_RA")
dir.create("data")
colorDict = c("2CLC"="#88535A",
              "Zscan4+"="#EF8264",
              "Pluri"="#F2CC8F")

colorAge = c("old"="#7a646a", 
             "young"="#88bce7") 

colorTools = c("SoloTE"="#A4DD9B", 
               "Stellarscope"="#4f5d93",
               "STARsolo"="#f0df93")

dataset_id <- "10xMouse_RA"

In [ ]:
theme_paper <- function(base_size = 17, base_family = "") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      plot.title = element_text(face = "bold", size = base_size + 2, hjust = 0.5),
      axis.title = element_text(size = base_size),
      axis.text  = element_text(size = base_size * 0.9),
      legend.title = element_text(size = base_size),
      legend.text  = element_text(size = base_size * 0.9),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      plot.margin = margin(10, 10, 10, 10)
    )
}
theme_set(theme_paper())

# Pre

In [ ]:
# Import Stellarscope object
objTE_stellarscope <- readRDS(paste0("data_",dataset_id,"/stellarscope_",dataset_id,"_seuratObj_251125.RDS"))
# Import SoloTE object
objTE_soloTE <- readRDS(paste0("data_",dataset_id,"/soloTE_",dataset_id,"_seuratObj_251124.RDS"))
# Import STARsolo object
objTE_STARsolo <- readRDS(paste0("data_",dataset_id,"/STARsolo_",dataset_id,"_seuratObj_251125.RDS"))

In [ ]:
objTE_stellarscope
objTE_soloTE
objTE_STARsolo

In [ ]:
conversionTable <- read.table("annotation/annotation_mm10_conversion_withAge.tsv") # generated with annotation_scripts/create_annotations_mouse.Rmd
head(conversionTable)

In [ ]:
# get TEs in common between Stellarscope and SoloTE

stellarscopeTEs <- Features(objTE_stellarscope)

table(stellarscopeTEs %in% conversionTable$stellarscopeID)

soloTEs <- Features(objTE_soloTE)
# remove "SoloTE" from the name of the TEs and subtitute "|" and "_" with "-"
soloTEs <- gsub("\\|", "-", soloTEs)
soloTEs <- gsub("\\_", "-", soloTEs)
soloTEs <- gsub("\\?", "", soloTEs)

table(soloTEs %in% conversionTable$soloteID)

TEsInStellarscope_SoloTE <- intersect( conversionTable$locusID[conversionTable$stellarscopeID %in% stellarscopeTEs],
                        conversionTable$locusID[conversionTable$soloteID %in% soloTEs] )
length(TEsInStellarscope_SoloTE)


In [ ]:
# get TEs in common between Stellarscope and STARsolo

stellarscopeTEs <- Features(objTE_stellarscope)

table(stellarscopeTEs %in% conversionTable$stellarscopeID)

STARsoloTEs <- Features(objTE_STARsolo)


table(STARsoloTEs %in% conversionTable$stellarscopeID)

TEsInStellarscope_STARsolo <- intersect( conversionTable$locusID[conversionTable$stellarscopeID %in% stellarscopeTEs],
                        conversionTable$locusID[conversionTable$stellarscopeID %in% STARsoloTEs] )
length(TEsInStellarscope_STARsolo)


In [ ]:
STARsoloTEs <- Features(objTE_STARsolo)
table(STARsoloTEs %in% conversionTable$stellarscopeID)

TEsInAll <- intersect( TEsInStellarscope_SoloTE, 
                       conversionTable$locusID[conversionTable$stellarscopeID %in% STARsoloTEs] )

length(TEsInAll)

In [ ]:
commonCellsAll <- intersect(intersect(Cells(objTE_stellarscope), Cells(objTE_soloTE)), Cells(objTE_STARsolo))
length(commonCellsAll)

In [ ]:
# # locus id of common TEs
tes_soloTE_conv <- conversionTable$locusID[match(Features(objTE_soloTE), conversionTable$soloteID)]
tes_stellarscope_conv <- conversionTable$locusID[match(Features(objTE_stellarscope), conversionTable$stellarscopeID)]
tes_starsolo_conv <- conversionTable$locusID[match(Features(objTE_STARsolo), conversionTable$stellarscopeID)]

commonTEs_conv <- intersect(intersect(tes_soloTE_conv, tes_stellarscope_conv), tes_starsolo_conv)
commonTEs_conv <- commonTEs_conv[!is.na(commonTEs_conv)]
length(commonTEs_conv)

In [ ]:

# ALL commmon TEs
# reconvert to original name
common_soloTE <- conversionTable$soloteID[match(commonTEs_conv, conversionTable$locusID)]
common_stellarscope <- conversionTable$stellarscopeID[match(commonTEs_conv, conversionTable$locusID)]
common_STARsolo <- conversionTable$stellarscopeID[match(commonTEs_conv, conversionTable$locusID)]

# extract matrix of common TEs
stellarscopeMatSub <- GetAssayData(objTE_stellarscope[common_stellarscope, commonCellsAll], layer = "data")
setdiff(common_stellarscope, rownames(stellarscopeMatSub))
stellarscopeMatSub <- stellarscopeMatSub[common_stellarscope, commonCellsAll] # to reorder

STARsoloMatSub <- GetAssayData(objTE_STARsolo[common_stellarscope, commonCellsAll], layer = "data")
STARsoloMatSub <- STARsoloMatSub[common_stellarscope, commonCellsAll] # to reorder

soloTEMatSub <- GetAssayData(objTE_soloTE[common_soloTE, commonCellsAll], layer = "data")
soloTEMatSub <- soloTEMatSub[common_soloTE, commonCellsAll] # to reorder

identical(colnames(stellarscopeMatSub), colnames(soloTEMatSub))
identical(colnames(STARsoloMatSub), colnames(soloTEMatSub))

matSubs <- list()
matSubs[["Stellarscope"]] <- stellarscopeMatSub
matSubs[["SoloTE"]] <- soloTEMatSub
matSubs[["STARsolo"]] <- STARsoloMatSub

gc()


In [ ]:
# matrices with common cells and TEs
mat_stellarscope <- as.matrix(stellarscopeMatSub)

mat_soloTE <- as.matrix(soloTEMatSub) 

mat_STARsolo <- as.matrix(STARsoloMatSub)


gc()

# Correlations

In [ ]:
avgExprCommonTEs <- rowMeans(mat_stellarscope)
# compute average expression in the cells where each TE is expressed
sumExprCommonTEs <- rowSums(mat_stellarscope)
nCellsExprTEs <- rowSums(mat_stellarscope>0)
avgExprPerExprCell <- sumExprCommonTEs/nCellsExprTEs

TEcorrsCommonTEs <- list()

# compute spearman correlation of stellarscope and soloTE counts
TEcorrsCommonTEs[["Stellarscope_SoloTE"]] <- sapply(1:nrow(mat_stellarscope), FUN = function(i){
  cor(mat_stellarscope[i,], mat_soloTE[i,], method="spearman")
})

# compute spearman correlation of stellarscope and STARsolo counts
TEcorrsCommonTEs[["Stellarscope_STARsolo"]] <- sapply(1:nrow(mat_stellarscope), FUN = function(i){
  cor(mat_stellarscope[i,], mat_STARsolo[i,], method="spearman")
})

# compute spearman correlation of stellarscope and STARsolo counts
TEcorrsCommonTEs[["STARsolo_SoloTE"]] <- sapply(1:nrow(mat_STARsolo), FUN = function(i){
  cor(mat_STARsolo[i,], mat_soloTE[i,], method="spearman")
})


In [ ]:
summary(TEcorrsCommonTEs[["Stellarscope_SoloTE"]])
summary(TEcorrsCommonTEs[["Stellarscope_STARsolo"]])
summary(TEcorrsCommonTEs[["STARsolo_SoloTE"]])

# Checkpoint

In [ ]:
#save.image(paste0("workspaces/afterCorrelations_wSTAR_thr2percCell_",dataset_id,".RData"))
load(paste0("workspaces/afterCorrelations_wSTAR_thr2percCell_",dataset_id,".RData"))
rm(mat_stellarscope)
rm(mat_soloTE)
rm(mat_STARsolo)
gc()


# Post

In [ ]:
options(repr.plot.width=5, repr.plot.height=4)

for(combo in names(TEcorrsCommonTEs)){
        
        tools <- unlist(strsplit(combo, "_"))
        show(
                ggplot() + aes(x=TEcorrsCommonTEs[[combo]]) + 
                        geom_density(color="#99918F", fill=alpha("#99918F",0.5), linewidth = 1) + 
                        ggtitle(paste0("Correlation between ", tools[1]), 
                                subtitle = paste0("and ", tools[2], " counts of common TEs")) +
                        xlab("Spearman cor. coef.") +
                        theme_minimal() + 
                        theme(text=element_text(size=14.5), 
                                plot.title = element_text(size=14.5, hjust=0.5), 
                                plot.subtitle = element_text(size=14.5, hjust=0.5)) 
        )
                ggsave(paste0("figures_",dataset_id,"/spearmanCorrelation_density_commonTEs_",combo,".pdf"), device = "pdf")
}

In [ ]:
# define age breaks for bins
breaks <- c(0, 2, 5, 10, 15, 25, 40, 60, Inf)

summary(conversionTable$mya)

conversionTable$age_bin <- cut(
  conversionTable$mya,
  breaks = breaks,
  labels = c("0-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40-60", "60+"),
  right = FALSE,
  include.lowest = TRUE
)

table(conversionTable$age_bin)

table(conversionTable$age_bin[is.na(conversionTable$mya)])

In [ ]:
options(repr.plot.width=5, repr.plot.height=4)

for(combo in names(TEcorrsCommonTEs)){
  tools <- unlist(strsplit(combo, "_"))

  df <- cbind(avgExprPerExprCell, TEcorrsCommonTEs[[combo]])
  df <- as.data.frame(df)
  colnames(df) <- c("avgExprPerExprCell", "TEcorrsCommonTEs")

  cat("Correlation between correlations and avg expression (", combo,") : ",
    cor(avgExprPerExprCell, TEcorrsCommonTEs[[combo]], method="spearman"),"\n")


  show(ggplot(df, aes(x=TEcorrsCommonTEs, y=avgExprPerExprCell)) + ggtitle(combo) +
        geom_point(alpha= 0.3))

  # divide by expr quartiles
  df$quartile <- "4"
  df$quartile[df$avgExprPerExprCell < quantile(df$avgExprPerExprCell, 0.75, na.rm=TRUE)] <- "3"
  df$quartile[df$avgExprPerExprCell < quantile(df$avgExprPerExprCell, 0.5, na.rm=TRUE)] <- "2"
  df$quartile[df$avgExprPerExprCell < quantile(df$avgExprPerExprCell, 0.25, na.rm=TRUE)] <- "1"

  show(ggplot(df, aes(x=quartile, y=TEcorrsCommonTEs)) + ggtitle(combo) +
    geom_violin(fill=alpha("#99918F",0.5)) + 
    theme(text=element_text(size=15)) +
    xlab("Stellarscope mean expression quartile") )

  # divide by correlation quartiles
  df$quartile <- "4"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.75, na.rm=TRUE)] <- "3"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.5, na.rm=TRUE)] <- "2"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.25, na.rm=TRUE)] <- "1"
  show(ggplot(df, aes(x=quartile, y=avgExprPerExprCell)) + 
    geom_violin(fill=alpha("#99918F",0.5)) + ggtitle(combo) +
    theme(text=element_text(size=15)) +
    scale_y_continuous(transform = "log") +
    xlab("Correlation quartile") )
}



## Correlation by age



In [ ]:
options(repr.plot.width=10, repr.plot.height=6)
library(ggbeeswarm)

dfsCorrelationAge <- list()

for(combo in names(TEcorrsCommonTEs)){
  tools <- unlist(strsplit(combo, "_"))

  df <- cbind(rownames(matSubs[[tools[1]]]), TEcorrsCommonTEs[[combo]])
  df <- as.data.frame(df)
  colnames(df) <- c("V1", "TEcorrsCommonTEs")

  df$ageClass <- conversionTable$ageClass[match(df$V1, conversionTable$stellarscopeID)]
  df$age_bin <- conversionTable$age_bin[match(df$V1, conversionTable$stellarscopeID)]
  df$mya <- conversionTable$mya[match(df$V1, conversionTable$stellarscopeID)]
  df$TEcorrsCommonTEs <- as.numeric(df$TEcorrsCommonTEs)
  cor(df$mya, df$TEcorrsCommonTEs, method="spearman")


  show(  ggplot(df, aes(x=age_bin, y=TEcorrsCommonTEs, fill=age_bin)) + 
        geom_violin(alpha = 0.8) + 
            stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5, color = "black") +
        theme_paper() +
        #scale_fill_manual(values=colorAge) +
        guides(fill = "none") +
        theme(text=element_text(size=18)) +
        xlab("Age bins (mya)") +
        ggtitle(combo)
  )

show(  ggplot(df, aes(x=age_bin, y=TEcorrsCommonTEs, color=age_bin)) + 
        geom_beeswarm(alpha = 0.7,cex = 0.2, size=0.5, width=0.5) +
        #geom_boxplot(outlier.shape = NA, width = 0.3, color = "gray40", fill=NA, alpha = 0.3) +
      stat_summary(
            fun = median, 
            geom = "crossbar", 
            width = 0.3, 
            color = "grey30", 
            linewidth = 0.5
      ) +
            stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
                  geom = "text", vjust = -0.5, size = 5, color = "black") +
        theme_paper() +
        #scale_fill_manual(values=colorAge) +
        guides(color = "none") +
        theme(text=element_text(size=18)) +
        xlab("Age bins (mya)") +
        ylab("Spearman cor. coef.") +
        ggtitle(combo)
  )
  ggsave(paste0("figures_",dataset_id,"/correlation_byAge_Bin_",combo,".pdf"), width=8, height=6)

  show(  ggplot(df, aes(x=ageClass, y=TEcorrsCommonTEs, fill=ageClass)) + 
        geom_violin(alpha = 0.5) + 
            stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5, color = "black") +
        theme_pubclean() +
        scale_fill_manual(values=colorAge) +
        theme(text=element_text(size=18)) +
        xlab("Age class") +
        ggtitle(combo)
  )

  # divide by correlation quartiles
  df$quartile <- "4"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.75, na.rm=TRUE)] <- "3"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.5, na.rm=TRUE)] <- "2"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.25, na.rm=TRUE)] <- "1"
  
  show(ggplot(df, aes(x=quartile, y=mya)) + 
        geom_violin(fill=alpha("#99918F",0.5)) + 
            stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5) +
        theme_pubclean() +
        theme(text=element_text(size=18)) +
        scale_y_continuous(transform = "log") +
        xlab("Correlation quartile") +
        ggtitle(combo)
  )

  dfsCorrelationAge[[combo]] <- df
}


In [ ]:
options(repr.plot.width=5, repr.plot.height=6)

dfsCorrelationAge <- list()

for(combo in names(TEcorrsCommonTEs)){
  tools <- unlist(strsplit(combo, "_"))

  df <- cbind(rownames(matSubs[[tools[1]]]), TEcorrsCommonTEs[[combo]])
  df <- as.data.frame(df)
  colnames(df) <- c("V1", "TEcorrsCommonTEs")

  df$ageClass <- conversionTable$ageClass[match(df$V1, conversionTable$stellarscopeID)]
  df$mya <- conversionTable$mya[match(df$V1, conversionTable$stellarscopeID)]
  df$TEcorrsCommonTEs <- as.numeric(df$TEcorrsCommonTEs)
  cor(df$mya, df$TEcorrsCommonTEs, method="spearman")

  options(repr.plot.width=5, repr.plot.height=6)
      
  show(  ggplot(df, aes(x=ageClass, y=TEcorrsCommonTEs, fill=ageClass)) + 
        geom_violin(jitter=TRUE, alpha = 0.5) + 
        theme_pubclean() +
        scale_fill_manual(values=colorAge) +
        theme(text=element_text(size=18)) +
        xlab("Age class") +
        ggtitle(combo)
  )

# with jitter
  # show(ggplot(df, aes(x=ageClass, y=TEcorrsCommonTEs, fill=ageClass)) + 
  #     geom_violin(jitter=TRUE, alpha = 0.5) + 
  #     geom_jitter(position = position_jitter(seed = 1, width = 0.2), size=0.1, alpha=0.2) +
  #     theme_pubclean() +
  #     scale_fill_manual(values=colorAge) +
  #     theme(text=element_text(size=18)) +
  #     xlab("Age class") +
  #     ggtitle(combo)
  # )


  # divide by correlation quartiles
  df$quartile <- "4"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.75, na.rm=TRUE)] <- "3"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.5, na.rm=TRUE)] <- "2"
  df$quartile[df$TEcorrsCommonTEs < quantile(df$TEcorrsCommonTEs, 0.25, na.rm=TRUE)] <- "1"

  show(ggplot(df, aes(x=quartile, y=mya)) + 
        geom_violin(fill=alpha("#99918F",0.5)) + 
        theme_pubclean() +
        theme(text=element_text(size=18)) +
        scale_y_continuous(transform = "log") +
        xlab("Correlation quartile") +
        ggtitle(combo)
  )

  dfsCorrelationAge[[combo]] <- df
}


In [ ]:
options(repr.plot.width=12, repr.plot.height=5)

ggplot(dfsCorrelationAge[["Stellarscope_SoloTE"]], aes(x=mya)) + 
  geom_density(alpha = 0.5) + 
  theme_light() +
  theme(text=element_text(size=18)) +
  xlab("Milion years of age") 
  
ggplot(dfsCorrelationAge[["Stellarscope_SoloTE"]], aes(x=mya, y=TEcorrsCommonTEs, color=ageClass)) + 
  geom_point(alpha = 0.5, ) + 
  scale_color_manual(values=colorAge) +
  theme_pubclean() +
  theme(text=element_text(size=18)) +
  xlab("Milion years of age") 

# this is only for the commonly detected TEs
# the pattern with tes common to SoloTE and Stellarscope was a bit different

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)

for(combo in names(dfsCorrelationAge)){

        tools <- unlist(strsplit(combo,"_"))

        show(ggplot(dfsCorrelationAge[[combo]]) + aes(x=TEcorrsCommonTEs, fill=ageClass) + 
                geom_density( linewidth = 0.5, alpha = 0.4 ) + 
                ggtitle(paste0("Correlation between ", tools[1]), 
                        subtitle = paste0("and ",tools[2]," counts of common TEs")) +
                xlab("Spearman cor. coef.") +
                scale_fill_manual(values=colorAge) +
                theme_minimal() + 
                theme(text=element_text(size=20), 
                        plot.title = element_text(size=20, hjust=0.5), 
                        plot.subtitle = element_text(size=20, hjust=0.5)) 
        )
        ggsave(paste0("figures_",dataset_id,"/correlation_byAge_",combo,".pdf"), width=7, height=5)

}


# Correlation by family



# Statistics on n features

In [ ]:
options(repr.plot.width=3, repr.plot.height=4)

GeomSplitViolin <- ggproto("GeomSplitViolin", GeomViolin, 
                           draw_group = function(self, data, ..., draw_quantiles = NULL) {
  data <- transform(data, xminv = x - violinwidth * (x - xmin), xmaxv = x + violinwidth * (xmax - x))
  grp <- data[1, "group"]
  newdata <- plyr::arrange(transform(data, x = if (grp %% 2 == 1) xminv else xmaxv), if (grp %% 2 == 1) y else -y)
  newdata <- rbind(newdata[1, ], newdata, newdata[nrow(newdata), ], newdata[1, ])
  newdata[c(1, nrow(newdata) - 1, nrow(newdata)), "x"] <- round(newdata[1, "x"])

  if (length(draw_quantiles) > 0 & !scales::zero_range(range(data$y))) {
    stopifnot(all(draw_quantiles >= 0), all(draw_quantiles <=
      1))
    quantiles <- ggplot2:::create_quantile_segment_frame(data, draw_quantiles)
    aesthetics <- data[rep(1, nrow(quantiles)), setdiff(names(data), c("x", "y")), drop = FALSE]
    aesthetics$alpha <- rep(1, nrow(quantiles))
    both <- cbind(quantiles, aesthetics)
    quantile_grob <- GeomPath$draw_panel(both, ...)
    ggplot2:::ggname("geom_split_violin", grid::grobTree(GeomPolygon$draw_panel(newdata, ...), quantile_grob))
  }
  else {
    ggplot2:::ggname("geom_split_violin", GeomPolygon$draw_panel(newdata, ...))
  }
})

geom_split_violin <- function(mapping = NULL, data = NULL, stat = "ydensity", position = "identity", ..., 
                              draw_quantiles = NULL, trim = TRUE, scale = "area", na.rm = FALSE, 
                              show.legend = NA, inherit.aes = TRUE) {
  layer(data = data, mapping = mapping, stat = stat, geom = GeomSplitViolin, 
        position = position, show.legend = show.legend, inherit.aes = inherit.aes, 
        params = list(trim = trim, scale = scale, draw_quantiles = draw_quantiles, na.rm = na.rm, ...))
}


In [ ]:
options(repr.plot.width=5, repr.plot.height=7)

nCountsSoloTE <- objTE_soloTE[,commonCellsAll]$nCount_RNA
nCountsStellarscope <- objTE_stellarscope[,commonCellsAll]$nCount_TE
nCountsSTARsolo <- objTE_STARsolo[,commonCellsAll]$nCount_TE

df <- as.data.frame(rbind(cbind(nCountsSoloTE,"SoloTE", sapply(strsplit(names(nCountsSoloTE), "_"),'[',1)),
                          cbind(nCountsStellarscope,"Stellarscope", sapply(strsplit(names(nCountsStellarscope), "_"),'[',1)),
                          cbind(nCountsSTARsolo,"STARsolo", sapply(strsplit(names(nCountsSTARsolo), "_"),'[',1))))

colnames(df) <- c("nCounts_TEs","Tool","CB")
df$Sample <- dataset_id
df$nCounts_TEs <- as.numeric(df$nCounts_TEs)

ggplot(df, aes(x=Sample, fill=Tool,  y=nCounts_TEs)) + 
  geom_violin(scale = "width", alpha = 0.8, color="grey20") +
  scale_fill_manual(values=colorTools)+
  ggtitle("") +
  xlab("") +
  ylab("tot TE counts per cell") +
  theme_minimal() + 
  theme(text=element_text(size=17), 
        plot.title = element_text(size=17, hjust=0.5), 
        plot.subtitle = element_text(size=17, hjust=0.5), 
        axis.text.x = element_text(size=15, angle=45, hjust=1)) 
ggsave(paste0("figures_",dataset_id,"/totCounts_SoloTEvsStellarscopevsSTARsolo.pdf"), device = "pdf", width=5, height=7)



In [ ]:
options(repr.plot.width=4, repr.plot.height=5)

nFeaturesSoloTE <- objTE_soloTE[,commonCellsAll]$nFeature_RNA
nFeaturesStellarscope <- objTE_stellarscope[,commonCellsAll]$nFeature_TE
nFeaturesSTARsolo <- objTE_STARsolo[,commonCellsAll]$nFeature_TE

df <- as.data.frame(rbind(cbind(nFeaturesSoloTE,"SoloTE", sapply(strsplit(names(nFeaturesSoloTE), "_"),'[',1)),
                          cbind(nFeaturesStellarscope,"Stellarscope", sapply(strsplit(names(nFeaturesStellarscope), "_"),'[',1)),
                          cbind(nFeaturesSTARsolo,"STARsolo", sapply(strsplit(names(nFeaturesSTARsolo), "_"),'[',1))))

colnames(df) <- c("nFeatures_TEs","Tool","Sample")
df$Sample <- "10xMouse_RA"
df$nFeatures_TEs <- as.numeric(df$nFeatures_TEs)
df$Sample <- factor(df$Sample, levels=unique(df$Sample))

ggplot(df, aes(x=Sample, fill=Tool, color=Tool, y=nFeatures_TEs)) + 
  geom_violin(scale = "width") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle("") +
  xlab("") +
  ylab("n TEs per cell") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=15, angle=30, hjust=1)) 
ggsave(paste0("figures_",dataset_id,"/totFeatures_SoloTEvsStellarscopevsSTARsolo_violin.pdf"), device = "pdf", width=4, height=5)


In [ ]:
objTE_stellarscope
mat <- GetAssayData(objTE_stellarscope, layer="counts")
sum(rowSums(mat > 0) >= 1)

In [ ]:
# save lists of TEs expressed in each sample
nTEs_soloTE <- list()
TEs_soloTE <- list()
for(ident in unique(objTE_soloTE$orig.ident)){
  mat <- GetAssayData(objTE_soloTE[,objTE_soloTE$orig.ident==ident], layer="counts")
  nTEs_soloTE[[ident]] <- sum(rowSums(mat > 0) >= 15)
  TEs_soloTE[[ident]] <- rownames(mat)[rowSums(mat > 0) >= 15]
}
TEs_soloTE_df <- stack(TEs_soloTE)
colnames(TEs_soloTE_df) <- c("locus", "orig.ident")

nTEs_stellarscope <- list()
TEs_stellarscope <- list()
for(ident in unique(objTE_stellarscope$orig.ident)){
  mat <- GetAssayData(objTE_stellarscope[,objTE_stellarscope$orig.ident==ident], layer="counts")
  nTEs_stellarscope[[ident]] <- sum(rowSums(mat > 0) >= 15)
  TEs_stellarscope[[ident]] <- rownames(mat)[rowSums(mat > 0) >= 15]
}
TEs_stellarscope_df <- stack(TEs_stellarscope)
colnames(TEs_stellarscope_df) <- c("locus", "orig.ident")

nTEs_star <- list()
TEs_star <- list()
for(ident in unique(objTE_STARsolo$orig.ident)){
  mat <- GetAssayData(objTE_STARsolo[,objTE_STARsolo$orig.ident==ident], layer="counts")
  nTEs_star[[ident]] <- sum(rowSums(mat > 0) >= 15)
  TEs_star[[ident]] <- rownames(mat)[rowSums(mat > 0) >= 15]
}
TEs_star_df <- stack(TEs_stellarscope)
colnames(TEs_star_df) <- c("locus", "orig.ident")

options(repr.plot.width=4, repr.plot.height=5)
# N TEs per tool
df <- rbind( cbind(unlist(nTEs_soloTE), names(nTEs_soloTE), rep("SoloTE",length(unique(objTE_soloTE$orig.ident)))),
             cbind(unlist(nTEs_stellarscope), names(nTEs_stellarscope), rep("Stellarscope",length(unique(objTE_stellarscope$orig.ident)))),
             cbind(unlist(nTEs_star), names(nTEs_star), rep("STARsolo",length(unique(objTE_STARsolo$orig.ident))))
)

colnames(df) <- c("n_TEs","Sample","Tool")
df <- as.data.frame(df)
df$n_TEs <- as.numeric(df$n_TEs)
df$Sample <- factor(df$Sample, levels=unique(df$Sample))
df 
ggplot(df, aes(x=Tool,fill=Tool, color=Tool, y=n_TEs)) + 
  geom_col(position = "dodge") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle("") +
  xlab("") +
  ylab("N. detected TEs") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=15, angle=45, hjust=1))
ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscopevsSTARsolo_barplot.pdf"), device = "pdf", width = 4, height=5)



In [ ]:
library(UpSetR)

options(repr.plot.width=10, repr.plot.height=5)

detectedList <- list(SoloTE = tes_soloTE_conv, Stellarscope = tes_stellarscope_conv, STARsolo = tes_starsolo_conv)

UpSetR::upset(fromList(detectedList), nintersects = 30,
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedList))+1000, #show.numbers = F,
            nsets=length(objList)+1,
            sets.x.label="N. detected loci",
            mainbar.y.label="Intersection size")
            ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscopevsSTARsolo_barplot.pdf"), device = "pdf", width = 10, height=5)


In [ ]:
options(repr.plot.width=5, repr.plot.height=5)

library(eulerr)

# remove NAs
sum(is.na(detectedList[["SoloTE"]])) # ?
detectedList[["SoloTE"]][is.na(detectedList[["SoloTE"]])] <- (1:sum(is.na(detectedList[["SoloTE"]])))
sum(is.na(detectedList[["SoloTE"]]))

# Convert lists to sets and compute intersections
fit <- euler(detectedList)

# Plot with proportional areas
plot(fit, 
     fills = colorTools, 
     alpha = 0.5,
     labels = list(font = 2, cex = 1.5),       # <-- increase set label size
     quantities = list(cex = 1.4),             # <-- increase count size
     main.cex = list(cex=2),
     edges = list(col = "white", lwd = 2),
     # labels = TRUE,
     # quantities = TRUE,
     main = "Overlap across detected TEs")

pdf("figures_10xMouse_RA/overlap_euler_withSTAR.pdf", width=5, height=5)
plot(fit, 
     fills = colorTools, 
     alpha = 0.5,
     labels = list(font = 2, cex = 1.5),       # <-- increase set label size
     quantities = list(cex = 1.4),             # <-- increase count size
     main.cex = list(cex=2),
     edges = list(col = "white", lwd = 2),
     # labels = TRUE,
     # quantities = TRUE,
     main = "Overlap across detected TEs")
dev.off()


In [ ]:
# Number of TEs detected above a certain threshold of cells ( 5% of the dataset)

cellNum <- round(length(Cells(objTE_soloTE))*0.05)
# save lists of TEs expressed in each sample
nTEs_soloTE <- list()
TEs_soloTE <- list()

for(ident in unique(objTE_soloTE$orig.ident)){
  mat <- GetAssayData(objTE_soloTE[,objTE_soloTE$orig.ident==ident], layer="counts")
  nTEs_soloTE[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_soloTE[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_soloTE_df <- stack(TEs_soloTE)
colnames(TEs_soloTE_df) <- c("locus", "orig.ident")

nTEs_stellarscope <- list()
TEs_stellarscope <- list()
for(ident in unique(objTE_stellarscope$orig.ident)){
  mat <- GetAssayData(objTE_stellarscope[,objTE_stellarscope$orig.ident==ident], layer="counts")
  nTEs_stellarscope[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_stellarscope[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_stellarscope_df <- stack(TEs_stellarscope)
colnames(TEs_stellarscope_df) <- c("locus", "orig.ident")

nTEs_star <- list()
TEs_star <- list()
for(ident in unique(objTE_STARsolo$orig.ident)){
  mat <- GetAssayData(objTE_STARsolo[,objTE_STARsolo$orig.ident==ident], layer="counts")
  nTEs_star[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_star[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_star_df <- stack(TEs_stellarscope)
colnames(TEs_star_df) <- c("locus", "orig.ident")

options(repr.plot.width=4, repr.plot.height=5)
# N TEs per tool
df <- rbind( cbind(unlist(nTEs_soloTE), names(nTEs_soloTE), rep("SoloTE",length(unique(objTE_soloTE$orig.ident)))),
             cbind(unlist(nTEs_stellarscope), names(nTEs_stellarscope), rep("Stellarscope",length(unique(objTE_stellarscope$orig.ident)))),
             cbind(unlist(nTEs_star), names(nTEs_star), rep("STARsolo",length(unique(objTE_STARsolo$orig.ident))))
)

colnames(df) <- c("n_TEs","Sample","Tool")
df <- as.data.frame(df)
df$n_TEs <- as.numeric(df$n_TEs)
df$Sample <- factor(df$Sample, levels=unique(df$Sample))
df 
ggplot(df, aes(x=Tool,fill=Tool, color=Tool, y=n_TEs)) + 
  geom_col(position = "dodge") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle(paste0("Loci detected in 5% of cells")) +
  xlab("") +
  ylab("N. detected TEs") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=15, angle=45, hjust=1))
ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscopevsSTARsolo_barplot_thr5percCells.pdf"), device = "pdf", width = 5, height=5)



In [ ]:
# Number of TEs detected above a certain threshold of cells ( 5% of the dataset)

cellNum <- round(length(Cells(objTE_soloTE))*0.02)
# save lists of TEs expressed in each sample
nTEs_soloTE <- list()
TEs_soloTE <- list()

for(ident in unique(objTE_soloTE$orig.ident)){
  mat <- GetAssayData(objTE_soloTE[,objTE_soloTE$orig.ident==ident], layer="counts")
  nTEs_soloTE[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_soloTE[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_soloTE_df <- stack(TEs_soloTE)
colnames(TEs_soloTE_df) <- c("locus", "orig.ident")

nTEs_stellarscope <- list()
TEs_stellarscope <- list()
for(ident in unique(objTE_stellarscope$orig.ident)){
  mat <- GetAssayData(objTE_stellarscope[,objTE_stellarscope$orig.ident==ident], layer="counts")
  nTEs_stellarscope[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_stellarscope[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_stellarscope_df <- stack(TEs_stellarscope)
colnames(TEs_stellarscope_df) <- c("locus", "orig.ident")

nTEs_star <- list()
TEs_star <- list()
for(ident in unique(objTE_STARsolo$orig.ident)){
  mat <- GetAssayData(objTE_STARsolo[,objTE_STARsolo$orig.ident==ident], layer="counts")
  nTEs_star[[ident]] <- sum(rowSums(mat > 0) >= cellNum)
  TEs_star[[ident]] <- rownames(mat)[rowSums(mat > 0) >= cellNum]
}
TEs_star_df <- stack(TEs_stellarscope)
colnames(TEs_star_df) <- c("locus", "orig.ident")

options(repr.plot.width=4, repr.plot.height=5)
# N TEs per tool
df <- rbind( cbind(unlist(nTEs_soloTE), names(nTEs_soloTE), rep("SoloTE",length(unique(objTE_soloTE$orig.ident)))),
             cbind(unlist(nTEs_stellarscope), names(nTEs_stellarscope), rep("Stellarscope",length(unique(objTE_stellarscope$orig.ident)))),
             cbind(unlist(nTEs_star), names(nTEs_star), rep("STARsolo",length(unique(objTE_STARsolo$orig.ident))))
)

colnames(df) <- c("n_TEs","Sample","Tool")
df <- as.data.frame(df)
df$n_TEs <- as.numeric(df$n_TEs)
df$Sample <- factor(df$Sample, levels=unique(df$Sample))
df 
ggplot(df, aes(x=Tool,fill=Tool, color=Tool, y=n_TEs)) + 
  geom_col(position = "dodge") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle(paste0("Loci detected in 2% of cells")) +
  xlab("") +
  ylab("N. detected TEs") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=15, angle=45, hjust=1))
ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscopevsSTARsolo_barplot_thr2percCells.pdf"), device = "pdf", width = 5, height=5)



In [ ]:
options(repr.plot.width=5, repr.plot.height=7)
TEs_soloTE_df$age <- conversionTable[match(TEs_soloTE_df$locus, conversionTable$soloteID),"ageClass"]
TEs_stellarscope_df$age <- conversionTable[match(TEs_stellarscope_df$locus, conversionTable$stellarscopeID),"ageClass"]
TEs_star_df$age <- conversionTable[match(TEs_star_df$locus, conversionTable$stellarscopeID),"ageClass"]

TEs_soloTE_df$tool <- "SoloTE"
TEs_stellarscope_df$tool <- "Stellarscope"
TEs_star_df$tool <- "STARsolo"
df <- rbind(TEs_soloTE_df,TEs_stellarscope_df,TEs_star_df)

df <- df[!is.na(df$age),] # why are there NAs? Didn't I remove TEs not in the annotation beforehand? 

ggplot(df, aes(x=tool, fill=age))+
  geom_bar(position="fill", alpha = 0.8) +
  scale_fill_manual(values=colorAge) + 
  ggtitle("") +
  xlab("") +
  ylab("% TEs") +
  theme_light() + 
  theme(text=element_text(size=20), 
        strip.text.x = element_text(size=22, hjust=0.5),
        strip.background = element_rect(fill="#7E7E7E"), 
        axis.text.x = element_text(size=18, angle=45, hjust=1))

ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscopevsSTARsolo_barplot_age_tool_celltype.pdf"), device = "pdf", width=5, height=7)


In [ ]:
options(repr.plot.width=8, repr.plot.height=7)
soloTEageFamily <- table(conversionTable[match(Features(objTE_soloTE), conversionTable$soloteID),"ageClass"],
                         conversionTable[match(Features(objTE_soloTE), conversionTable$soloteID),"class"])
soloTEageFamilyOld <- cbind(soloTEageFamily["old",],colnames(soloTEageFamily),"old","SoloTE")
colnames(soloTEageFamilyOld) <- c("nTEs","superfamily","age","tool")
soloTEageFamilyYoung <- cbind(soloTEageFamily["young",],colnames(soloTEageFamily),"young","SoloTE")
colnames(soloTEageFamilyYoung) <- c("nTEs","superfamily","age","tool")
soloTEageFamily <- rbind(soloTEageFamilyOld, soloTEageFamilyYoung)
soloTEageFamily <- as.data.frame(soloTEageFamily)
soloTEageFamily$nTEs <- as.numeric(soloTEageFamily$nTEs)

stellarscopeageFamily <- table(conversionTable[match(Features(objTE_stellarscope), conversionTable$stellarscopeID),"ageClass"],
                         conversionTable[match(Features(objTE_stellarscope), conversionTable$stellarscopeID),"class"])
stellarscopeageFamilyOld <- cbind(stellarscopeageFamily["old",],colnames(stellarscopeageFamily),"old","Stellarscope")
colnames(stellarscopeageFamilyOld) <- c("nTEs","superfamily","age","tool")
stellarscopeageFamilyYoung <- cbind(stellarscopeageFamily["young",],colnames(stellarscopeageFamily),"young","Stellarscope")
colnames(stellarscopeageFamilyYoung) <- c("nTEs","superfamily","age","tool")
stellarscopeageFamily <- rbind(stellarscopeageFamilyOld, stellarscopeageFamilyYoung)
stellarscopeageFamily <- as.data.frame(stellarscopeageFamily)
stellarscopeageFamily$nTEs <- as.numeric(stellarscopeageFamily$nTEs)


ageFamilyDf <- rbind(soloTEageFamily, stellarscopeageFamily)
ageFamilyDf$superfamily <- gsub(ageFamilyDf$superfamily, pattern="\\?", replacement="")

ageFamilyDf <- ageFamilyDf[ageFamilyDf$nTEs>0,]


ggplot(ageFamilyDf, aes(x=tool, fill=superfamily, y=nTEs)) +
  facet_wrap(~age) +
  geom_col(position="fill") +
  #scale_fill_manual(values=adjustcolor(c("#5E50A1","#3787BC","#67C1A3","#AADCA2","#E6F596"), alpha.f = 0.8)) + 
  scale_fill_brewer(palette = "Spectral", direction = -1)+
  #palette = "Spectral", direction = -1)+
  # scale_color_manual(values=c("#81B29A","#9181A6"))+
  ggtitle("") +
  xlab("") +
  ylab("% TEs") +
  theme_light() + 
  theme(text=element_text(size=20), 
        strip.text.x = element_text(size=22, hjust=0.5),
        strip.background = element_rect(fill="#3d405b"),  #"#e78063"
        axis.text.x = element_text(size=18, angle=45, hjust=1))

ggsave(paste0("figures_",dataset_id,"/nTEs_SoloTEvsStellarscope_barplot_age_order.pdf"), device = "pdf")



In [ ]:
options(repr.plot.width=4, repr.plot.height=6)

nOldSoloTE <- table(conversionTable[match(Features(objTE_soloTE), conversionTable$soloteID),"ageClass"])["old"]
nYoungSoloTE <- table(conversionTable[match(Features(objTE_soloTE), conversionTable$soloteID),"ageClass"])["young"]
nOldStellarscope <- table(conversionTable[match(Features(objTE_stellarscope), conversionTable$stellarscopeID),"ageClass"])["old"]
nYoungStellarscope <- table(conversionTable[match(Features(objTE_stellarscope), conversionTable$stellarscopeID),"ageClass"])["young"]
nOldSTARsolo <- table(conversionTable[match(Features(objTE_STARsolo), conversionTable$stellarscopeID),"ageClass"])["old"]
nYoungSTARsolo <- table(conversionTable[match(Features(objTE_STARsolo), conversionTable$stellarscopeID),"ageClass"])["young"]


table(conversionTable[match(Features(objTE_stellarscope), conversionTable$stellarscopeID),"ageClass"])

df <- rbind( c(nOldSoloTE , nOldSoloTE / (nOldSoloTE + nYoungSoloTE), "old", "SoloTE"),
             c(nYoungSoloTE, nYoungSoloTE / (nOldSoloTE + nYoungSoloTE), "young", "SoloTE"),
             c(nOldStellarscope, nOldStellarscope / (nOldStellarscope + nYoungStellarscope), "old", "Stellarscope"),
             c(nYoungStellarscope, nYoungStellarscope  / (nOldStellarscope + nYoungStellarscope), "young", "Stellarscope"),
             c(nOldSTARsolo, nOldSTARsolo / (nOldSTARsolo + nYoungSTARsolo), "old", "STARsolo"),
             c(nYoungSTARsolo, nYoungSTARsolo  / (nOldSTARsolo + nYoungSTARsolo), "young", "STARsolo")
             )
df <- as.data.frame(df)


colnames(df) <- c("n_TEs","n_TEs_prop", "AgeClass", "Tool")
df$n_TEs <- as.numeric(df$n_TEs)
df$n_TEs_prop <- as.numeric(df$n_TEs_prop)
df$n_TEs_perc <- as.numeric(df$n_TEs_prop) * 100



ggplot(df[df$AgeClass=="young",], aes(x=Tool, fill=Tool, color=Tool, y=n_TEs_perc)) + 
  geom_col(position = "stack") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle("") +
  xlab("") +
  ylab("% young TEs") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=18, angle=45, hjust=1))
ggsave(paste0("figures_",dataset_id,"/barplot_percYoungTEs.pdf"), device = "pdf", width = 4, height=6)

ggplot(df[df$AgeClass=="young",], aes(x=Tool, fill=Tool, color=Tool, y=n_TEs)) + 
  geom_col(position = "stack") +
  scale_fill_manual(values=colorTools)+
  scale_color_manual(values=colorTools)+
  ggtitle("") +
  xlab("") +
  ylab("N. young TEs") +
  theme_minimal() + 
  theme(text=element_text(size=18), 
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        axis.text.x = element_text(size=18, angle=45, hjust=1))
ggsave(paste0("figures_",dataset_id,"/barplot_nYoungTEs.pdf"), device = "pdf", width = 4, height=6)


# Celltype purity

In [ ]:
# copy cell type annotation to STARsolo and stellarscope objects

identical(Cells(objTE_STARsolo), Cells(objTE_soloTE))
objTE_STARsolo$celltype <- objTE_soloTE$celltype

identical(Cells(objTE_stellarscope), Cells(objTE_soloTE))
objTE_stellarscope$celltype <- objTE_soloTE$celltype


In [ ]:
objTE_stellarscope <- FindNeighbors(objTE_stellarscope, dims = 1:13, k.param = 20, return.neighbor = T)

purity <- sapply(Cells(objTE_stellarscope), FUN = function(cell){
  n_cells <- TopNeighbors(objTE_stellarscope@neighbors$RNA.nn, cell, n = 20)
  celltypes <- objTE_stellarscope@meta.data[n_cells,'celltype']
  label <- objTE_stellarscope@meta.data[cell,'celltype']
  perc_label <- length(celltypes[celltypes==label]) / 20
})
df <- as.data.frame(cbind(purity, "Stellarscope"))
colnames(df) <- c("purity", "tool")
df$celltype <- objTE_stellarscope@meta.data[rownames(df),'celltype']
df$purity <- as.numeric(df$purity)



objTE_soloTE <- FindNeighbors(objTE_soloTE, dims = 1:10, k.param = 20, return.neighbor = T)

purity <- sapply(Cells(objTE_soloTE), FUN = function(cell){
  n_cells <- TopNeighbors(objTE_soloTE@neighbors$RNA.nn, cell, n = 20)
  celltypes <- objTE_soloTE@meta.data[n_cells,'celltype']
  label <- objTE_soloTE@meta.data[cell,'celltype']
  perc_label <- length(celltypes[celltypes==label]) / 20
})
df2 <- as.data.frame(cbind(purity, "SoloTE"))
colnames(df2) <- c("purity", "tool")
df2$celltype <- objTE_soloTE@meta.data[rownames(df2),'celltype']
df2$purity <- as.numeric(df2$purity)

objTE_STARsolo <- FindNeighbors(objTE_STARsolo, dims = 1:10, k.param = 20, return.neighbor = T)

purity <- sapply(Cells(objTE_STARsolo), FUN = function(cell){
  n_cells <- TopNeighbors(objTE_STARsolo@neighbors$RNA.nn, cell, n = 20)
  celltypes <- objTE_STARsolo@meta.data[n_cells,'celltype']
  label <- objTE_STARsolo@meta.data[cell,'celltype']
  perc_label <- length(celltypes[celltypes==label]) / 20
})
df3 <- as.data.frame(cbind(purity, "STARsolo"))
colnames(df3) <- c("purity", "tool")
df3$celltype <- objTE_STARsolo[,Cells(objTE_soloTE)]@meta.data[rownames(df2),'celltype']
df3$purity <- as.numeric(df3$purity)


df_purity <- rbind(df,df2,df3)


In [ ]:
# colorTools = c("SoloTE"="#A4DD9B", #6fc69d",
#                "Stellarscope"="#4f5d93",
#                "STARsolo"="#f0df93")

df_purity$tool <- factor(df_purity$tool, levels = c("STARsolo","Stellarscope","SoloTE"))

options(repr.plot.width=9.5, repr.plot.height=6)

ggplot(df_purity) + aes(y=purity, x=celltype, fill=tool) + geom_boxplot(linewidth = 0.8, alpha=0.8, outlier.size=0.7, outlier.alpha = 0.5) + 
  ggtitle("Purity of cell type labels among KNN-graph neighbors") +
  scale_fill_manual(values=colorTools)+
  theme_minimal() + 
  theme(text=element_text(size=20),
        plot.title = element_text(size=23, hjust=0.5), 
        legend.text = element_text(size=20),
        axis.text.x = element_text(size=20, angle=35, hjust=1),
        axis.text.y = element_text(size=20, hjust=1)) 
ggsave(paste0("figures_", dataset_id,"/celltype_purity_knn_soloTEvsStellarscopevsSTAR_noPlatelet.pdf"), 
      width=9.5, height=6 )